In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (Plisson et al.)

This notebook curates the **Plisson et al.** collection by integrating multiple peptide sources and exporting clean, task-specific datasets.\
 The pipeline assembles two primary positive sets, separately tracks sequences with reported modifications, \
 and also exports additional **unlabeled** sequences included in the source package. All outputs follow the project’s standard schema and include \
 duplicate consistency checks plus metadata.

 - **Toxic effect / endpoint:** hemolytic
- **Source:** Plisson et al.
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads multiple raw inputs** provided by the source package (CSVs and FASTA files), including:
  - a consolidated activity table (`complete_df_with_activity.csv`) from which hemolytic positives are extracted,
  - additional curated sets (e.g., `HAMP.fasta`, APD-derived tables, Hemolytik positive/negative FASTA folders),
  - a larger activity table (`total_APD.csv`) and a version containing modifications (`total_APD_modified.csv`),
  - a collection of sequences without explicit labels (multiple FASTA/CSV files combined into an *unlabeled* set).
- **Builds task-specific datasets**:
  - **Hemolytic** positives are assembled by concatenating hemolysis-positive subsets from all relevant inputs.
- **Creates additional auxiliary datasets**:
  - **Modified subsets**: hemolytic sequences extracted from the “modified” table.
  - **Unlabeled subset**: sequences gathered from several files that do not provide class labels (`label = 2`).
- **Checks duplicated sequences** independently for:
  - hemolytic,
  - modified hemolytic
  - unlabeled set.
  Duplicates are collapsed when consistent; conflicting cases are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv`
  - `modified_hemolytic_dataset.csv`
  - `detected_unlabel_sequences.csv`
  - `metadata.json`

In [2]:
name_source = "Plisson et al."
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_complete_df_with_activity = pd.read_csv(f"{PATH_INPUT}/{name_source}/complete_df_with_activity.csv")

In [4]:
df_hemolytic_complete = (
    df_complete_df_with_activity[df_complete_df_with_activity["Hemolytic"] == 1]
    .rename(columns ={"Sequence":"sequence", "Hemolytic": "label"})
    [["sequence", "label"]]
)

In [5]:
df_hamp = (read_fasta_doc(f"{PATH_INPUT}/{name_source}/HAMP.fasta")
           .assign(label=1)
           [["sequence", "label"]])

In [6]:
df_apd = pd.read_csv(f"{PATH_INPUT}/{name_source}/hemolytic_APD.csv")

In [7]:
df_hemolytic_apd = (
    df_apd[df_apd["Hemolytic"] == 1]
    .rename(columns ={"Sequence":"sequence", "Hemolytic": "label"})
    [["sequence", "label"]]
)

In [8]:
df_hemolytik = pd.concat([
    read_fasta_doc(os.path.join(folder, file)).assign(label=0 if 'hemo_negative' in folder else 1)[["sequence", "label"]]
    for folder in [f"{PATH_INPUT}/{name_source}/Hemolytik_datasets/hemo_negative", 
                   f"{PATH_INPUT}/{name_source}/Hemolytik_datasets/hemo_positive"] 
    for file in os.listdir(folder)])

In [9]:
df_total = pd.read_csv(f"{PATH_INPUT}/{name_source}/total_APD.csv")

In [10]:
df_hemolytic_total = (
    df_total
    .assign(
        label=lambda d: d["Activity"]
        .str.contains("hemolytic", case=False, na=False)
        .astype(int)
    )
    .rename(columns={"Sequence": "sequence"})
    .loc[lambda d: d["label"] == 1, ["sequence", "label"]]
    .reset_index(drop=True)
)

In [11]:
df_total_modified = pd.read_csv(f"{PATH_INPUT}/{name_source}/total_APD_modified.csv")

In [12]:
df_hemolytic_modified = (
    df_total_modified
    .assign(
        label=lambda d: d["Activity"]
        .str.contains("hemolytic", case=False, na=False)
        .astype(int)
    )
    .rename(columns={"Sequence": "sequence"})
    .loc[lambda d: d["label"] == 1, ["sequence", "label"]]
    .reset_index(drop=True)
)

In [13]:
df_unlabel = (pd.concat([
        read_fasta_doc(f"{PATH_INPUT}/{name_source}/HemoPI/HemoPI1.fasta"),
        read_fasta_doc(f"{PATH_INPUT}/{name_source}/HemoPI/HemoPI2.fasta"),
        read_fasta_doc(f"{PATH_INPUT}/{name_source}/HemoPI/HemoPI3.fasta"),
        read_fasta_doc(f"{PATH_INPUT}/{name_source}/RPS.fasta"),
        read_fasta_doc(f"{PATH_INPUT}/{name_source}/selected_inliers.fasta"),
        pd.read_csv(f"{PATH_INPUT}/{name_source}/selected_inliers_sequences.csv")
        ], ignore_index=True)
        .assign(label=2)
        [["sequence", "label"]]
    )

- Concatenate dataset

In [14]:
df_hemolytic = pd.concat([
    df_hemolytic_complete,
    df_hamp,
    df_hemolytic_apd,
    df_hemolytik,
    df_hemolytic_total],
    ignore_index=True
)
df_hemolytic.shape

(4987, 2)

- Checking duplicates

In [15]:
df_remove_duplicated_hemolytic, df_errors_hemolytic, df_unique_hemolytic = processing_duplicated(df_hemolytic, group_seq="sequence", sort_key="label")

In [16]:
df_remove_duplicated_hemolytic_mod, df_errors_hemolytic_mod, df_unique_hemolytic_mod = processing_duplicated(df_hemolytic_modified, group_seq="sequence", sort_key="label")

In [17]:
df_remove_duplicated_unlabel, df_errors_unlabel, df_unique_unlabel = processing_duplicated(df_unlabel, group_seq="sequence", sort_key="label")

In [18]:
df_full_hemolytic = pd.concat([df_unique_hemolytic, df_remove_duplicated_hemolytic])

In [19]:
df_full_hemolytic_mod = pd.concat([df_unique_hemolytic_mod, df_remove_duplicated_hemolytic_mod])

In [20]:
df_full_unlabel = pd.concat([df_unique_unlabel, df_remove_duplicated_unlabel])

- Working with metada

In [21]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [22]:
raw_total_sequences = (
    len(df_complete_df_with_activity)
    + len(df_hamp)
    + len(df_apd)
    + len(df_hemolytik)
    + len(df_total)
    + len(df_total_modified)
    + len(df_unlabel)
)

In [23]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": int(len(df_full_hemolytic) + len(df_full_unlabel)),
    "number_of_positive_sequences": int((df_full_hemolytic["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full_hemolytic["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors_hemolytic),
    "number_of_modified_sequences" : len(df_full_hemolytic_mod),
    "number_of_erroneous_modified_sequences" : len(df_errors_hemolytic_mod),
    "number_of_unlabel_sequences" : len(df_full_unlabel),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'MIT',
 'year of publication': 2020,
 'last update date': datetime.datetime(2020, 8, 24, 0, 0),
 'download date': Timestamp('2025-08-12 00:00:00'),
 'file format': 'csv;fasta',
 'peptide property': 'anti gram positive, anti gram negative, anti gram variable, antifungal, antibiofilm, anti mammalian cells, anticancer, antiviral, anti HIV, antimicrobial surfaceImmobilized, antiparasitic, anti MRSA, enzyme inhibitor, chemotactic, insecticidal, antioxidant, spermicidal, hemolytic, candidacidal;hemolytic, toxic;hemolytic,antifungal,antiviral,spermicidal,anticancer,antibiofilm,antiparasitic,chemotactic,enzyme inhibitor,antioxidant, anti HIV, insecticidal, anti MRSA, wound healing, antimalaria;antifungal,candidacidal,anticancer,wound healing,antiviral, hemolytic,spermicidal,insecticidal,chemotactic, anti gram variable, anti gram positive, anti gram negative, antimalaria, anti HIV, antiparasitic, antidiabetic, anti MRSA, antiin

- Exporting data

In [24]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [25]:
df_full_hemolytic.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_hemolytic_mod.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/modified_hemolytic_dataset.csv", index=False)
df_full_unlabel.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_unlabel_sequences.csv", index=False)